# Question 3 — Quantum Fourier Transformation

**HW1, Group 18 — DD2360/FDD3360 Applied GPU Programming**

This notebook profiles the supplied, algorithmically unmodified `qft_bench_naive.c` on the QuEST CUDA backend. The saved GPU query identifies a **Tesla T4 (compute capability 7.5)**. The existing experimental outputs below have been preserved; the explanatory cells were added during a review and do not represent new GPU runs.

## Completion and evidence

| Item | Evidence available | Remaining work before submission |
|---|---|---|
| 1. GPU setup and 24-qubit execution | GPU query; successful 24-qubit output in the profiling cell; troubleshooting recorded in `report.md` | Add a screenshot of the actual 24-qubit terminal/notebook output. The previously referenced `images/q3_1_24q_output.png` is still absent. |
| 2. Memory at 28 qubits | Saved benchmark output, device memory sampling, and process peak RSS | None for the reported measurements; preserve the logs when repeating. |
| 3. Top three kernels at 24 qubits | Saved `cuda_gpu_kern_sum` table and five timed repetitions | None for the reported table. |
| 4. Kernel-to-gate mapping | Kernel definitions, API implementation, and expected instance counts | Record the exact QuEST revision when repeating; the original checkout SHA was not saved. |
| 5. CPU launch and GPU execution | Saved `cudaLaunchKernel` statistics; 24-qubit GUI screenshot with kernel and Correlation ID 806 | Capture the matched CPU `cudaLaunchKernel` event/details to make the individual launch relationship directly visible. |
| 6. Scaling to 16 and 28 qubits | Saved benchmark results and both GUI screenshots | Screenshots show a short region rather than the boundaries of a whole repetition; keep that distinction in the interpretation. |

## Execution environment and timing definitions

Run the code cells in a **Linux GPU environment**, such as a Colab T4 runtime. Upload the supplied `qft_bench_naive.c` to `/content/` before building. Nsight Systems CLI must be installed in the GPU environment; its desktop GUI is used separately to inspect downloaded `.nsys-rep` files. Keep this notebook beside the repository's `images/` folder for the local image links to resolve. Colab may require uploading the images or displaying downloaded reports separately.

The benchmark initializes a plus state, performs **one warm-up QFT**, and then performs **five timed QFTs**. For each timed repetition, initialization and its synchronization happen before `t0`; the measured interval includes `QFT(...)` and the final `syncQuESTEnv()`. Thus `best / mean` reports synchronized host elapsed time **per QFT**, including launch overhead and waiting for the GPU. It excludes state creation, per-repetition initialization, and the final probability check.

Nsight Systems records the **whole process**, so its kernel summary includes warm-up, initialization and validation as well as timed work. Its `Time (%)` is a share of summed GPU kernel time, not a share of application wall time. See the [NVIDIA statistical report definitions](https://docs.nvidia.com/nsight-systems/AnalysisGuide/index.html#cuda-gpu-kern-sum).


## 1. GPU setup, troubleshooting, and 24-qubit run

The query below reports `Tesla T4, 7.5`, so CMake uses `CMAKE_CUDA_ARCHITECTURES=75` (use the architecture of your actual GPU). The benchmark must print **GPU backend**, not CPU backend.

The existing group report records an initial CMake working-directory problem: commands were run from `/content` rather than the QuEST source directory. The original failing output is not saved in this notebook. The build cell below now specifies both `-S /content/QuEST` and `-B /content/QuEST/build`, avoiding reliance on notebook directory state, and separates the CMake flags clearly. No benchmark algorithm or timing code was changed.

The first two code cells are the original environment/discovery cells. On a new runtime, clone once; when rerunning an existing runtime, skip the clone if `/content/QuEST` already exists.


In [ ]:
!nvidia-smi --query-gpu=name,compute_cap   
!git clone https://github.com/QuEST-Kit/QuEST.git

name, compute_cap
Tesla T4, 7.5
Cloning into 'QuEST'...
remote: Enumerating objects: 17409, done.
remote: Counting objects: 100% (1737/1737), done.
remote: Compressing objects: 100% (1023/1023), done.
remote: Total 17409 (delta 1253), reused 750 (delta 714), pack-reused 15672 (from 4)
Receiving objects: 100% (17409/17409), 19.35 MiB | 25.90 MiB/s, done.
Resolving deltas: 100% (11826/11826), done.


In [11]:
!find /content -name "qft_bench_naive.c" 2>/dev/null
%cd /content/QuEST
!ls

/content/qft_bench_naive.c
/content/QuEST
AUTHORS.txt  CMakeLists.txt  examples	  quest      tests
cmake	     docs	     LICENCE.txt  README.md  utils


In [ ]:
%%bash
set -euo pipefail
test -f /content/qft_bench_naive.c
test -f /content/QuEST/CMakeLists.txt
cp /content/qft_bench_naive.c /content/QuEST/qft_bench_naive.c
cmake -S /content/QuEST -B /content/QuEST/build \
  -DQUEST_ENABLE_CUDA=ON \
  -DCMAKE_CUDA_ARCHITECTURES=75 \
  -DCMAKE_CUDA_FLAGS="-lineinfo" \
  -DUSER_SOURCE_NAMES=qft_bench_naive.c \
  -DUSER_OUTPUT_EXE_NAME=qft_naive
cmake --build /content/QuEST/build -j4
/content/QuEST/build/qft_naive 24 5


### Result and screenshot requirement

The build cell originally contained no saved output and has not been executed during this review. The later, preserved **24-qubit Nsight Systems run** does establish successful CUDA execution:

```text
QFT benchmark: 24 qubits, 5 reps, GPU backend, 0.25 GiB state vector
  rep 0:  385.017 ms
  rep 1:  385.014 ms
  rep 2:  385.023 ms
  rep 3:  384.936 ms
  rep 4:  384.999 ms
  total prob    : 1.000000000000
  gates per QFT : 312
  best / mean   : 384.936 / 384.998 ms
```

This is a transcription of saved output from the profiling cell below, not a new run. `report.md` also records a separate 24-qubit result of **385.173 / 385.250 ms**; its original standalone output is not saved in this notebook, so these two runs must not be mixed.

**Still required:** save a screenshot of the actual 24-qubit output as `images/q3_1_24q_output.png`. A typeset transcription does not replace the screenshot requested in the assignment. The probability of 1 is a normalization sanity check, not a complete verification of all QFT amplitudes.

### Optional environment record for future runs

The next cell has not been run here. It records the current environment when you execute it; it cannot recover the exact revision or tool versions used for the historical outputs.


In [ ]:
%%bash
set -euo pipefail
cd /content/QuEST
mkdir -p q3_artifacts
{
  date -u
  git rev-parse HEAD
  sha256sum qft_bench_naive.c
  nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv
  nvcc --version
  cmake --version
  nsys --version
} | tee q3_artifacts/environment.txt


## 2. GPU and system memory at 28 qubits

The original measurement code below runs `qft_naive 28 5` without Nsight Systems. It samples device memory using `nvidia-smi` while `/usr/bin/time -v` records the benchmark process's maximum resident set size. The `time` package installation is only needed if `/usr/bin/time` is missing.


In [19]:
!apt-get update -qq && apt-get install -y time

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  time
0 upgraded, 1 newly installed, 0 to remove and 71 not upgraded.
Need to get 45.2 kB of archives.
After this operation, 129 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 time amd64 1.9-0.2build1 [45.2 kB]
Fetched 45.2 kB in 0s (160 kB/s)
Selecting previously unselected package time.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../time_1.9-0.2build1_amd64.deb ...
Unpacking time (1.9-0.2build1) ...
Setting up time (1.9-0.2build1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [20]:
%%bash

EXE=/content/QuEST/build/qft_naive

echo "=== GPU memory before ==="
nvidia-smi --query-gpu=memory.used --format=csv,noheader

echo "=== System memory before ==="
free -h

echo "=== Running 28-qubit QFT ==="

/usr/bin/time -v "$EXE" 28 5 > /tmp/qft28_output.txt 2> /tmp/qft28_memory.txt &
PID=$!

GPU_PEAK=0

while kill -0 "$PID" 2>/dev/null; do
    USED=$(nvidia-smi \
        --query-gpu=memory.used \
        --format=csv,noheader,nounits | head -n 1)

    if [[ "$USED" =~ ^[0-9]+$ ]] && (( USED > GPU_PEAK )); then
        GPU_PEAK=$USED
    fi

    sleep 0.1
done

wait "$PID"
STATUS=$?

echo
echo "=== QFT benchmark output ==="
cat /tmp/qft28_output.txt

echo
echo "=== GPU memory peak ==="
echo "$GPU_PEAK MiB"

echo
echo "=== System memory peak (process RSS) ==="
grep "Maximum resident set size" /tmp/qft28_memory.txt

echo
echo "=== Exit status ==="
echo "$STATUS"

=== GPU memory before ===
0 MiB
=== System memory before ===
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.1Gi       8.1Gi       2.3Mi       3.8Gi        11Gi
Swap:             0B          0B          0B
=== Running 28-qubit QFT ===

=== QFT benchmark output ===
QFT benchmark: 28 qubits, 5 reps, GPU backend, 4.00 GiB state vector
  rep 0: 8200.340 ms
  rep 1: 8193.259 ms
  rep 2: 8185.796 ms
  rep 3: 8178.002 ms
  rep 4: 8169.742 ms
  total prob    : 1.000000000000

  gates per QFT : 420
  best / mean   : 8169.742 / 8185.428 ms
  time per gate : 19451.768 us

=== GPU memory peak ===
4203 MiB

=== System memory peak (process RSS) ===
	Maximum resident set size (kbytes): 183492

=== Exit status ===
0


### Results and interpretation

| Quantity | Saved measurement | Meaning |
|---|---:|---|
| GPU memory before execution | 0 MiB | Device-wide baseline on the selected GPU |
| Highest observed GPU memory during execution | **4203 MiB = 4.1045 GiB** | Maximum sampled `memory.used`, including allocations and CUDA overhead |
| Peak system memory of benchmark process | **183492 KiB = 179.19 MiB** | Maximum RSS from `/usr/bin/time -v`; not total machine RAM usage |
| Exit status | 0 | Successful benchmark execution |

Double-precision complex amplitudes occupy 16 bytes each. The state-vector storage is therefore

\[
M(n)=2^n\times16\ \text{bytes},\qquad M(28)=4\ \text{GiB}=4096\ \text{MiB}.
\]

The sampled GPU allocation exceeds the theoretical state vector by **107 MiB**. CUDA runtime/context overhead and other allocations can account for this difference; this measurement does not attribute every byte. `nvidia-smi --query-gpu=memory.used` is device-wide, so it can include other GPU processes. Here the initial baseline was 0 MiB; an isolated GPU makes attribution more reliable.

The loop sleeps for 0.1 s **in addition to** the time taken by `nvidia-smi`, so the sampling interval is greater than 100 ms and short peaks may be missed. RSS measures resident host pages and does not include GPU device memory. `free -h` supplies machine context only; it is not the benchmark's peak memory measurement. These host and device measurements have different scopes and need not add up to the state-vector size.

This memory run printed best/mean **8169.742 / 8185.428 ms** for 28 qubits. The Nsight Systems run in Question 6 is a separate measurement and printed slightly different times.


## 3. Top three GPU kernels at 24 qubits

The following original cell uses only Nsight Systems and the supplied benchmark. It records all process activity, including one warm-up and five timed repetitions. It expects the working directory `/content/QuEST`, established by the earlier `%cd` cell. On a repeat run, rename or remove the old output reports first to avoid overwriting evidence unintentionally.


In [21]:
!nsys profile --trace=cuda,osrt --stats=true -o qft_24q ./build/qft_naive 24 5

QFT benchmark: 24 qubits, 5 reps, GPU backend, 0.25 GiB state vector
  rep 0:  385.017 ms
  rep 1:  385.014 ms
  rep 2:  385.023 ms
  rep 3:  384.936 ms
  rep 4:  384.999 ms
  total prob    : 1.000000000000

  gates per QFT : 312
  best / mean   : 384.936 / 384.998 ms
  time per gate : 1233.768 us
Generating '/tmp/nsys-report-0e9d.qdstrm'
[1/7] [========================100%] qft_24q.nsys-rep
[2/7] [========================100%] qft_24q.sqlite
[3/7] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)     Min (ns)   Max (ns)    StdDev (ns)        Name     
 --------  ---------------  ---------  ------------  -------------  --------  -----------  ------------  --------------
     94.0    2,930,589,514         38  77,120,776.7  100,134,934.5   328,296  325,024,146  59,070,980.5  poll          
      5.9      183,378,732        508     360,981.8       14,712.0     1,259   18,291,169   1,266,552.5  ioctl         
      0.1        1,754,001      

### `cuda_gpu_kern_sum` results

The saved table uses nanoseconds. Total times below are converted to ms by dividing by \(10^6\), and averages to µs by dividing by \(10^3\).

| Rank | Kernel base name (template parameter in saved output) | GPU time share | Total time (ms) | Instances | Average duration (µs) |
|---|---|---:|---:|---:|---:|
| 1 | `kernel_statevec_anyCtrlOneTargDiagMatr_sub<1>` | 80.8% | 1891.753348 | 1656 | 1142.3631 |
| 2 | `kernel_statevec_anyCtrlOneTargDenseMatr_subA<0>` | 14.3% | 335.207961 | 144 | 2327.8331 |
| 3 | `kernel_statevec_anyCtrlSwap_subA<0>` | 4.1% | 96.363784 | 72 | 1338.3859 |

The phase-shift kernel is the main hotspot **by accumulated GPU time**. A Hadamard kernel invocation is slower on average, but phase shifts occur much more often. These totals cover six QFT executions; they are not times for one QFT or for just the five timed repetitions. Additional CUB kernels serve initialization and the probability calculation.


## 4. Map kernels to quantum gates

| Gate in the benchmark | QuEST API | Observed GPU kernel | Calls per QFT at 24 qubits | Expected instances for 6 QFTs |
|---|---|---|---:|---:|
| Controlled phase shift | `applyTwoQubitPhaseShift(q, c, t, angle)` | `kernel_statevec_anyCtrlOneTargDiagMatr_sub<1>` | \(24\times23/2=276\) | 1656 |
| Hadamard | `applyHadamard(q, t)` | `kernel_statevec_anyCtrlOneTargDenseMatr_subA<0>` | 24 | 144 |
| Final bit-order reversal by SWAP | `applySwap(q, i, n-1-i)` | `kernel_statevec_anyCtrlSwap_subA<0>` | 12 | 72 |

The [QuEST API implementation](https://github.com/QuEST-Kit/QuEST/blob/main/quest/src/api/operations.cpp) constructs a controlled diagonal matrix for a two-qubit phase shift, the Hadamard dense matrix, and a swap operation. In [gpu_kernels.cuh](https://github.com/QuEST-Kit/QuEST/blob/main/quest/src/gpu/gpu_kernels.cuh), the corresponding kernels multiply selected amplitudes by diagonal factors, mix amplitude pairs through a 2×2 matrix, and exchange amplitudes. The observed template arguments distinguish one control from no controls. [gpu_subroutines.cpp](https://github.com/QuEST-Kit/QuEST/blob/main/quest/src/gpu/gpu_subroutines.cpp) contains their GPU dispatch.

This mapping is supported by both the source operations and the exact instance counts. The source links point to upstream `main`; the original experiment's revision was not saved, so inspect the actual experimental checkout when reproducing.

For arbitrary \(n\), the benchmark executes

\[
G(n)=n+\frac{n(n-1)}{2}+\left\lfloor\frac n2\right\rfloor.
\]

Thus \(G(16)=144\), \(G(24)=312\), and \(G(28)=420\), matching the saved benchmark outputs. At 16 qubits the expected phase/Hadamard/SWAP instance counts are 720/96/48, and at 28 qubits they are 2268/168/84; both match the saved profiles.


## 5. CPU launch calls and the 24-qubit GUI timeline

The saved `cuda_api_sum` table identifies **`cudaLaunchKernel`** as the CPU-side kernel-launch API (1881 calls over the whole 24-qubit process, including utility kernels). `cudaDeviceSynchronize` waits for completion; it does not launch the QFT gates. Its long CPU duration largely reflects waiting for asynchronous GPU work.

Open `qft_24q.nsys-rep` in the Nsight Systems GUI, expand the `qft_naive` thread's **CUDA API** row and the **CUDA HW / Kernels** rows, and zoom into a QFT sequence between initialization and the final synchronization. To identify a timed repetition, skip the first warm-up sequence and inspect one of the next five; an arbitrary short crop alone does not establish the repetition number. Select a GPU gate kernel and navigate to its correlated `cudaLaunchKernel` event, checking the Correlation ID and launching thread. Repeat for the diagonal, dense-matrix, and SWAP kernels.

![24-qubit QFT timeline](images/q3_5_24q_timeline.png)

The existing screenshot directly shows `kernel_statevec_anyCtrlOneTargDenseMatr_subA` (Hadamard): **2.303 ms GPU execution**, launching thread **23889**, **Correlation ID 806**, and displayed launch latency **14.095 µs**. Latency is distinct from both GPU execution time and the CPU API call duration. The screenshot does **not** expose the duration of the matching CPU API event, so the earlier report's **14.039 µs CPU launch duration** is not independently supported by this image.

**Evidence to finish:** save a view/details panel displaying `cudaLaunchKernel` with Correlation ID 806 beside the selected GPU event. If documenting all three gate types, record the corresponding event IDs for the other two as well. The current crop shows a short section, not the start and end of an entire QFT repetition. See the [Nsight Systems GUI guide](https://docs.nvidia.com/nsight-systems/UserGuide/index.html) for timeline navigation.


## 6. Repeat at 16 and 28 qubits

The two original cells below record the unmodified benchmark at 16 and 28 qubits, each with five timed repetitions plus one warm-up.


In [22]:
!nsys profile --trace=cuda,osrt --stats=true -o qft_16q ./build/qft_naive 16 5

QFT benchmark: 16 qubits, 5 reps, GPU backend, 0.00 GiB state vector
  rep 0:    2.202 ms
  rep 1:    2.099 ms
  rep 2:    1.741 ms
  rep 3:    1.741 ms
  rep 4:    1.773 ms
  total prob    : 1.000000000000

  gates per QFT : 144
  best / mean   : 1.741 / 1.911 ms
  time per gate : 12.088 us
Generating '/tmp/nsys-report-c840.qdstrm'
[1/7] [========================100%] qft_16q.nsys-rep
[2/7] [========================100%] qft_16q.sqlite
[3/7] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)     Med (ns)    Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  -----------  --------  -----------  ------------  ----------------------
     73.6      524,177,776         14  37,441,269.7  8,242,973.5     1,548  323,061,398  84,931,538.6  poll                  
     25.7      182,880,086        501     365,030.1     17,571.0     1,030   21,739,329   1,319,031.4  ioctl                 
      0.3     

In [23]:
!nsys profile --trace=cuda,osrt --stats=true -o qft_28q ./build/qft_naive 28 5

QFT benchmark: 28 qubits, 5 reps, GPU backend, 4.00 GiB state vector
  rep 0: 8195.689 ms
  rep 1: 8195.787 ms
  rep 2: 8191.217 ms
  rep 3: 8184.746 ms
  rep 4: 8177.909 ms
  total prob    : 1.000000000000

  gates per QFT : 420
  best / mean   : 8177.909 / 8189.069 ms
  time per gate : 19471.211 us
Generating '/tmp/nsys-report-0beb.qdstrm'
[1/7] [========================100%] qft_28q.nsys-rep
[2/7] [========================100%] qft_28q.sqlite
[3/7] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls     Avg (ns)         Med (ns)        Min (ns)       Max (ns)     StdDev (ns)            Name         
 --------  ---------------  ---------  ---------------  ---------------  -------------  -------------  ------------  ----------------------
     50.8   49,881,857,800        507     98,386,307.3    100,136,061.0          1,516    291,605,394  16,260,963.6  poll                  
     48.9   48,002,508,914         12  4,000,209,076.2  4,000,082,828.0  4,000,019,536  4

### Total QFT timing printed by the benchmark

| Qubits | Printed rep 0–4 times (ms) | Best (ms/QFT) | Mean (ms/QFT) | Sum of printed five reps (ms) |
|---|---|---:|---:|---:|
| 16 | 2.202, 2.099, 1.741, 1.741, 1.773 | 1.741 | 1.911 | 9.556 |
| 28 | 8195.689, 8195.787, 8191.217, 8184.746, 8177.909 | 8177.909 | 8189.069 | 40945.348 |

`best / mean` is elapsed time for a **complete QFT per repetition**, not a kernel duration. The last column is calculated from the rounded per-repetition values printed in the saved output; the benchmark itself does not print the sum, and the warm-up is excluded from it. Rounding explains why five times the printed mean can differ slightly from the sum. Both runs print total probability `1.000000000000`.

### Typical kernel duration seen on the timelines

| Qubits | Selected kernel in the supplied screenshot | Duration visible in GUI | Same kernel's average in saved statistics |
|---|---|---:|---:|
| 16 | `kernel_statevec_anyCtrlOneTargDenseMatr_subA` (Hadamard) | **13.728 µs** | 13.5664 µs |
| 28 | `kernel_statevec_anyCtrlOneTargDiagMatr_sub` (phase shift) | **18.048 ms** | 18.1531318 ms |

These are representative individual events, not averages measured from screenshots. The screenshots select **different gate types**, so their two event durations are not a like-for-like scaling comparison. For the same phase-shift kernel, the statistics give **6.9480 µs** at 16 qubits versus **18.1531318 ms** at 28 qubits; this is supplementary statistical evidence rather than a 16-qubit phase-kernel tooltip measurement.

![16-qubit QFT timeline; Hadamard GPU duration 13.728 microseconds](images/q3_6_16q_timeline.png)

*16 qubits: the selected Hadamard tooltip shows 13.728 µs, Correlation ID 470. The state vector occupies 1 MiB; `0.00 GiB` in the benchmark output is rounding, not zero allocation.*

![28-qubit QFT timeline; phase-shift GPU duration 18.048 milliseconds](images/q3_6_28q_timeline.png)

*28 qubits: the selected phase-shift tooltip shows 18.048 ms, Correlation ID 1024; the state vector occupies 4 GiB. The displayed 36.872 ms latency is not the kernel duration.*

### Scaling interpretation

The mean complete-QFT time grows by about **4285×** between these recorded runs. The state vector grows by \(2^{28-16}=4096\), and gate count grows from 144 to 420. Larger vectors increase per-gate memory work; launch overhead, GPU utilization, memory access patterns, and different gate costs also matter. The traces show visible gaps between small kernels at 16 qubits and much longer, nearly continuous GPU work at 28 qubits. This supports an overhead-sensitive small case and a much more expensive large-vector case, but Nsight Systems timing alone does not measure achieved memory bandwidth or prove a specific bottleneck.

## Preserve evidence when reproducing

The original memory cell stores logs in `/tmp`; Colab runtime files are temporary. Download those logs and `qft_16q.nsys-rep`, `qft_24q.nsys-rep`, and `qft_28q.nsys-rep`, together with environment details and screenshots, before the runtime disconnects. The raw report files are not included in this repository, so the present review checked saved text and screenshots rather than re-opening raw captures. No CUDA/Nsight execution was performed on the local Windows machine during this review.
